# 01 · Define and check the programs

 **Question:** Do compiled transformers implement the intended sequence algorithms,
 and where does each intermediate variable get written?

 Start here, then follow [02 · Trace](02_trace.ipynb) and
 [03 · Geometry](03_geometry.ipynb). Each notebook can also run independently.

## Configure
Use Python 3.12 and the RaspFormer kernel. Change the settings and focus program below.

In [ ]:
import sys
from pathlib import Path

# Run from the repository root or any folder inside it.
start = Path.cwd().resolve()
PROJECT_ROOT = next(
    (root for root in (start, *start.parents) if (root / "raspformer").is_dir()),
    None,
)
if PROJECT_ROOT is None:
    raise RuntimeError("Open this notebook from inside the RaspFormer repository.")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))


import pandas as pd
from IPython.display import Markdown, display

from raspformer.compiler import CompilerConfig, run_study, schedule_table, summary_table
from raspformer.programs import PROGRAM_DESCRIPTIONS, build_programs

In [ ]:
config = CompilerConfig(
    vocab=tuple(range(-2, 6)),
    max_seq_len=6,
    causal=False,
    bos="BOS",
    pad="PAD",
    mlp_exactness=100,
    rel_tol=1e-6,
    abs_tol=1e-6,
)
samples = [
    [1, 2, 1],
    [-2, 0, 3, 1],
    [5],
    [0, 0, 0, 0],
    [5, 4, 3, 2, 1, 0],
]

OUTPUT_ROOT = PROJECT_ROOT / "results" / "programs"
FOCUS_PROGRAM = "P05_histogram"

## Run

 Build fresh RASP expressions, compile them, and compare five sequences per program
 with the RASP evaluator. The samples cover repetition, negatives, a singleton,
 and the maximum sequence length. Each run gets a new directory.

In [ ]:
PROGRAMS = build_programs(config.max_seq_len)
display(
    pd.DataFrame(
        [{"program": name, "behavior": PROGRAM_DESCRIPTIONS[name]} for name in PROGRAMS]
    )
)

In [ ]:
result = run_study(PROGRAMS, config, samples, OUTPUT_ROOT)
evidence = result["evidence"]
print(f"Saved evidence: {result['run_dir']}")
display(Markdown(summary_table(evidence["programs"])))

## Inspect

 A lane is a coordinate in the residual stream. The writer schedule tells you which
 attention or MLP block is allocated to write a variable. For `SelectorWidth`, the
 attention block starts the calculation and the following MLP writes its output.

In [ ]:
display(Markdown(schedule_table(evidence["variables"], FOCUS_PROGRAM)))
checks = pd.DataFrame(evidence["checks"])
display(checks[checks.program == FOCUS_PROGRAM].drop(columns="program"))

In [ ]:
print(f"{len(evidence['programs'])} programs; {len(evidence['checks'])} sampled checks passed.")
print(f"Devices: {evidence['settings']['devices']}")
print(f"Tracr revision: {evidence['settings']['tracr_commit']}")
print(f"Manifest and readable schedule: {result['run_dir']}")

## Interpret

 A completed run establishes agreement for these sampled inputs. It does not prove
 correctness for every allowed sequence. Writer schedules are compiler allocations,
 not measurements of activation onset.

 **Next:** [Trace the circuit calculations](02_trace.ipynb).